# Energy Fitting -- charge-to-energy calibration

Reads the 2D histogram `EnergyReconstruction.ipynb` wrote (true cluster energy
vs reco cluster charge, a `TH2D` in a `.root` file) and fits the energy-vs-charge
relation, so a measured charge can be turned into an energy.

**Three models, all through the origin** -- zero collected charge must mean zero
deposited energy, so none has a constant term:

| model | form | behaviour at large charge |
|---|---|---|
| linear | `E = a*Q` | keeps rising at the same rate |
| quadratic | `E = a*Q + b*Q²` | bends, but a downward parabola eventually turns over and predicts energy *decreasing* with charge |
| saturating | `E = a*Q / (1 + c*Q)` | rises with slope `a` at small charge and flattens to a plateau `a/c` -- the shape the data actually has |

**Rebinning**: the charge axis is merged `REBIN_X` bins at a time before
fitting. Coarser bins put more pairs into each charge column, which is what the
profile needs -- a column's mean energy is only as good as the number of pairs
behind it, and the sparse high-charge tail has columns holding one or two.

**Fit ranges**: the notebook runs every entry in `FIT_CONFIGS`, each into its own
subdirectory:

| config | charge | energy |
|---|---|---|
| `charge_to_5e7ADC` | ≤ `5e7` ADC | no limit |

The energy axis is deliberately left unrestricted: an upper cut on `y` truncates
the very distribution whose mean the profile is measuring, biasing every charge
column that reaches past it.

Bins are selected by their *center*, so a bin straddling a window edge is in or
out as a whole -- what is plotted is exactly what was fitted.

**How the fit is done**: weighted least squares over the filled bins, each bin
entering at its center with weight = its count, so every pair counts once.

**How the three are judged** -- three different questions, all worth asking:

- **χ²/ndf against the profile.** For each charge column, the weighted mean
  energy and its standard error give a point with a real uncertainty; χ² is
  measured against those. A χ² built from the raw bin scatter instead would just
  be measuring the *width* of the energy distribution at fixed charge, which no
  smooth curve can ever fit, and would look terrible for any model.
- **F-test**, for the *nested* pairs only. Linear sits inside both of the others
  (`b=0` gives the quadratic back, `c=0` the saturating one), so an F-test says
  whether the second parameter is earned in each case. Quadratic and saturating
  are separate two-parameter families and **cannot** be compared this way.
- **AIC**, which ranks all three on one scale -- the only tool here that can put
  quadratic and saturating side by side. ΔAIC under 2 is noise, over 10 decisive.

A model can win every comparison and still describe the data badly, so the
verdict reads them together, and the residual plot shows a wrong *shape*
directly -- a fit of the wrong form leaves a trend in its residuals even when its
χ²/ndf looks tolerable.

**Each fitted model is an energy estimator** -- feed it a cluster's charge and it
returns that cluster's reconstructed energy. Every model therefore also gets:

- `reco_vs_true_energy/` -- 2D histogram of true cluster energy (y) against the
  model's reco energy (x), with the `reco = true` diagonal drawn. Offset from
  that line is a bias; splaying away from it at one end is the wrong shape there.
- `energy-resolution/` -- the 1D `(reco - true)/true` distribution with a
  **Gaussian fitted to its core**, refitted within ±2σ so the long tails do not
  inflate the quoted width. `resolution_results.txt` ranks the models by bias
  (μ) and by resolution (σ) separately: a biased but narrow estimator can be
  fixed with a scale factor, a wide one cannot.

All fitting and drawing lives in `fit_energy_calibration.py` (this directory).
ROOT I/O uses **uproot**, not PyROOT (the ROOT installs on this machine conflict
and PyROOT does not import); it reads the same file with no ROOT at all.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from datetime import datetime

# This notebook lives in EnergyReconstruction/, next to fit_energy_calibration.py.
NB_DIR = Path.cwd()
if NB_DIR.name != "EnergyReconstruction":
    NB_DIR = NB_DIR / "EnergyReconstruction"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

from fit_energy_calibration import (
    read_hist2d, rebin_hist2d, hist2d_to_points, fit_model, profile_points,
    chi_square_against_profile, compare_fits, rank_fits, format_model,
    saturation_plateau, draw_fit, draw_residuals, write_fit_results,
    draw_reco_vs_true_energy, draw_energy_resolution, write_resolution_results,
    DEFAULT_HIST_NAME, ALL_MODELS,
)

print(f"Notebook directory: {NB_DIR}")


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================
# The 2D histogram to fit -- written by EnergyReconstruction.ipynb at job level.
# all_true_clusters/ (neutrino + cosmic) has the statistics and the charge range
# that make a calibration fit meaningful; true_neutrino_clusters/ holds only the
# neutrino pairs, far fewer and bunched at low charge.
# INPUT_BASE_DIR is the tree EnergyReconstruction.ipynb wrote (Before/After
# beam-window cut); the fit output goes at the TOP of that tree, not inside the
# timestamped run directory it reads from -- the run directory is that job's
# output, and a fit is a separate piece of work that may be redone many times
# against the same job.
INPUT_BASE_DIR = (NB_DIR / "multi_file_plots_charge_light_matching"
                         / "EnergyReconstruction_BeforeTimeWindowCut")

ROOT_FILE = (INPUT_BASE_DIR / "combined_apa_20260804_005916_allfiles"
                            / "job_summary" / "all_true_clusters"
                            / "true_cluster_energy_vs_reco_charge_hist2d_job_Combined.root")

HIST_NAME = DEFAULT_HIST_NAME     # 'true_energy_vs_reco_charge'

# ============================================================================
# REBINNING
# ============================================================================
# Merge this many charge bins together before fitting. The histogram is written
# with 2e6 ADC bins, so REBIN_X = 2 gives 4e6 ADC bins -- fewer, better-populated
# charge columns, which is what the profile and its uncertainties need.
REBIN_X = 2
REBIN_Y = 1           # energy bins left as written (100 MeV)

# ============================================================================
# FIT RANGES
# ============================================================================
# Every config here is fitted, plotted and tabulated into its own subdirectory.
# Bins are selected by their CENTER, and both axes start at 0 rather than at the
# data's first bin: the models are constrained through the origin, so the region
# near zero is part of what they have to describe.
#
# 'extrapolate_to' draws a second plot with the x axis running out that far and
# the curves dashed beyond the fitted region, to see where each model would take
# a charge it was never fitted on. None skips that plot -- there is nothing to
# extrapolate into for a fit that already covers the whole axis.
FIT_CONFIGS = [
    {'name': 'charge_to_5e7ADC',
     'x_min': 0.0, 'x_max': 5e7, 'y_min': 0.0, 'y_max': None,
     'extrapolate_to': 1.5e8,
     'description': 'charge <= 5e7 ADC, full energy range'},
]

# Output directory for the fit plots and fit_results.txt: a timestamped
# directory at the top of INPUT_BASE_DIR, so repeated fits accumulate side by
# side without ever writing into the histogram job's own output.
OUTPUT_DIR = INPUT_BASE_DIR / f"energy_fit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Input ROOT file : {ROOT_FILE}")
print(f"Histogram       : {HIST_NAME}")
print(f"Rebin           : x by {REBIN_X}, y by {REBIN_Y}")
print(f"Fit configs     : {', '.join(config['name'] for config in FIT_CONFIGS)}")
print(f"Output directory: {OUTPUT_DIR}")

if not ROOT_FILE.exists():
    raise FileNotFoundError(
        f"{ROOT_FILE} does not exist -- run EnergyReconstruction.ipynb first, or point "
        f"ROOT_FILE at an existing job_summary/*/*.root")


In [ ]:
# ============================================================================
# READ THE HISTOGRAM AND REBIN
# ============================================================================
hist_raw = read_hist2d(ROOT_FILE, HIST_NAME)

print(f"As written: {hist_raw['counts'].shape[0]} x {hist_raw['counts'].shape[1]} bins, "
      f"{hist_raw['n_entries']:.0f} entries")
print(f"  charge axis: {hist_raw['x_edges'][0]:.4g} to {hist_raw['x_edges'][-1]:.4g} ADC "
      f"({hist_raw['x_edges'][1] - hist_raw['x_edges'][0]:.4g} per bin)")
print(f"  energy axis: {hist_raw['y_edges'][0]:.4g} to {hist_raw['y_edges'][-1]:.4g} MeV "
      f"({hist_raw['y_edges'][1] - hist_raw['y_edges'][0]:.4g} per bin)")

hist = rebin_hist2d(hist_raw, x_factor=REBIN_X, y_factor=REBIN_Y)

print(f"\nAfter rebinning x by {REBIN_X}, y by {REBIN_Y}: "
      f"{hist['counts'].shape[0]} x {hist['counts'].shape[1]} bins, {hist['n_entries']:.0f} entries")
print(f"  charge bin width: {hist['x_edges'][1] - hist['x_edges'][0]:.4g} ADC")
print(f"  energy bin width: {hist['y_edges'][1] - hist['y_edges'][0]:.4g} MeV")

# Rebinning only ever moves entries between bins, never loses them -- if these
# ever disagree the merge is broken, rather than the histogram being unusual.
assert hist['n_entries'] == hist_raw['n_entries'], "rebinning lost entries"


In [ ]:
# ============================================================================
# FIT EVERY CONFIG: linear, quadratic and saturating, all through the origin
# ============================================================================
file_label = f"{ROOT_FILE.parent.parent.parent.name} / {ROOT_FILE.parent.name}"
results = {}

for config in FIT_CONFIGS:
    print("=" * 78)
    print(f"{config['name']}  ({config['description']})")
    print("=" * 78)

    x, y, w = hist2d_to_points(hist, x_max=config['x_max'], y_max=config['y_max'],
                               x_min=config['x_min'], y_min=config['y_min'])
    window = {'x_min': config['x_min'],
              'x_max': config['x_max'] if config['x_max'] is not None else float(hist['x_edges'][-1]),
              'y_min': config['y_min'],
              'y_max': config['y_max'] if config['y_max'] is not None else float(hist['y_edges'][-1]),
              'n_entries': float(w.sum()), 'n_bins': int(len(x))}
    print(f"  {window['n_entries']:.0f} entries in {window['n_bins']} filled bins "
          f"({window['n_entries'] / hist['n_entries']:.1%} of the histogram)")

    # The profile: one point per charge column, its weighted mean energy and the
    # standard error of that mean. This is what chi2/ndf is measured against.
    y_bin_width = float(hist['y_edges'][1] - hist['y_edges'][0])
    profile = profile_points(x, y, w, y_bin_width=y_bin_width)
    print(f"  profile: {len(profile[0])} charge columns with enough entries to carry an uncertainty")

    fits = [fit_model(x, y, w, model) for model in ALL_MODELS]
    for fit in fits:
        chi_square_against_profile(fit, profile)

    # F-tests, valid only for the NESTED pairs: linear sits inside both of the
    # others. Quadratic vs saturating are separate two-parameter families -- the
    # AIC ranking is what compares those.
    linear_fit, quadratic_fit, saturating_fit = fits
    comparisons = [compare_fits(linear_fit, quadratic_fit),
                   compare_fits(linear_fit, saturating_fit)]
    ranking = rank_fits(fits)
    best_by_chi2 = min(fits, key=lambda f: abs(f['chi2_per_ndf'] - 1.0))

    for fit in fits:
        print(f"  {fit['model'].upper():<11s} {format_model(fit)}")
        for name, value, error in zip(fit['param_names'], fit['params'], fit['param_errors']):
            print(f"      {name} = {value:.6g} +/- {error:.3g}")
        plateau = saturation_plateau(fit)
        if plateau is not None:
            print(f"      plateau a/c = {plateau:.1f} MeV")
        if fit['model'] == 'quadratic' and fit['params'][1] < 0:
            print(f"      turns over at Q = {abs(fit['params'][0] / (2 * fit['params'][1])):.4g} ADC")
        print(f"      chi2/ndf = {fit['chi2']:.2f} / {fit['ndf']} = {fit['chi2_per_ndf']:.3f}"
              f"   sigma = {fit['sigma']:.1f} MeV   AIC = {fit['aic']:.1f}")

    for comparison in comparisons:
        print(f"  F-test {comparison['simpler']} vs {comparison['richer']}: {comparison['reason']}"
              f"  ->  prefers {comparison['preferred']}")
    print("  AIC ranking: " + ",  ".join(f"{model} ({delta:+.2f})"
                                         for model, aic, delta in ranking['ranking']))
    print(f"  best by AIC: {ranking['best']}   best by chi2/ndf: {best_by_chi2['model']}\n")

    # ------------------------------------------------------------------
    # PLOTS + RESULTS TABLE, one subdirectory per config
    # ------------------------------------------------------------------
    config_dir = OUTPUT_DIR / config['name']
    title_suffix = f"({config['description']}, x rebinned by {REBIN_X})"

    draw_fit(hist, fits, config_dir, profile=profile,
             x_min=window['x_min'], x_max=config['x_max'],
             y_min=window['y_min'], y_max=config['y_max'],
             filename='energy_calibration_fit.png',
             title=f'Charge-to-Energy Calibration Fit {title_suffix}',
             file_label=file_label)

    if config['extrapolate_to']:
        draw_fit(hist, fits, config_dir, profile=profile,
                 x_min=window['x_min'], x_max=config['x_max'],
                 y_min=window['y_min'], y_max=config['y_max'],
                 extrapolate_to=config['extrapolate_to'],
                 filename='energy_calibration_fit_extrapolated.png',
                 title=f'Charge-to-Energy Calibration Fit, extrapolated {title_suffix}',
                 file_label=file_label)

    draw_residuals(fits, profile, config_dir,
                   title=f'Fit Residuals (profile - fit) {title_suffix}',
                   file_label=file_label)
    write_fit_results(fits, comparisons, hist, window, config_dir, ranking=ranking)

    # ------------------------------------------------------------------
    # EACH MODEL AS AN ENERGY ESTIMATOR: reco energy vs true energy, and the
    # fractional error it makes. Same pairs as the fit above -- the model is
    # being judged on the data it was fitted to, which is what makes these a
    # statement about the model's SHAPE rather than about extrapolation.
    # ------------------------------------------------------------------
    resolutions = []
    for fit in fits:
        draw_reco_vs_true_energy(fit, x, y, w, config_dir / 'reco_vs_true_energy',
                                 file_label=file_label)
        resolutions.append(
            draw_energy_resolution(fit, x, y, w, config_dir / 'energy-resolution',
                                   file_label=file_label))
    write_resolution_results(resolutions, fits, config_dir / 'energy-resolution')

    print("  energy resolution ((reco - true)/true, Gaussian core fit):")
    for resolution in resolutions:
        print(f"      {resolution['model']:<11s} mu = {resolution['mean']:+.4f} "
              f"+/- {resolution['mean_error']:.4f}   sigma = {resolution['sigma']:.4f} "
              f"+/- {resolution['sigma_error']:.4f}   chi2/ndf = {resolution['chi2_per_ndf']:.2f}")
    converged = [r for r in resolutions if r['converged'] and np.isfinite(r['sigma'])]
    if converged:
        print(f"      smallest |bias|: {min(converged, key=lambda r: abs(r['mean']))['model']}   "
              f"smallest sigma: {min(converged, key=lambda r: r['sigma'])['model']}")
    else:
        print("      no Gaussian fit converged")

    results[config['name']] = {'fits': fits, 'ranking': ranking, 'window': window,
                               'profile': profile, 'comparisons': comparisons,
                               'resolutions': resolutions}
    print(f"  output: {config_dir}\n")


In [ ]:
# ============================================================================
# SUMMARY ACROSS FIT RANGES
# ============================================================================
# The same three models over different ranges. Comparing one model's numbers
# BETWEEN ranges is meaningless -- different data, different chi2 -- but the
# RANKING within each range says whether the preferred shape depends on how much
# of the saturating region the fit was allowed to see.
for name, result in results.items():
    print(f"{name}: {result['window']['n_entries']:.0f} entries")
    best_aic = min(f['aic'] for f in result['fits'])
    for fit in result['fits']:
        print(f"   {fit['model']:<11s} chi2/ndf = {fit['chi2_per_ndf']:8.3f}   "
              f"dAIC = {fit['aic'] - best_aic:8.2f}   {format_model(fit)}")
    print(f"   -> best by AIC: {result['ranking']['best']}")
    for resolution in result['resolutions']:
        print(f"      {resolution['model']:<11s} resolution: mu = {resolution['mean']:+.4f}, "
              f"sigma = {resolution['sigma']:.4f}")
    converged = [r for r in result['resolutions'] if r['converged'] and np.isfinite(r['sigma'])]
    print(f"   -> smallest sigma: {min(converged, key=lambda r: r['sigma'])['model'] if converged else 'n/a'}\n")

print(f"All fit output in: {OUTPUT_DIR}")
